# Data Transformation

In [1]:
import os
import pandas as pd
from decimal import Decimal
import re

def to_decimal_or_none(x):
    return Decimal(x) if x and x.lower() != 'nan' else None

# Load all the CSVs into DataFrames
# Convert monetary columns to Decimal
orders = pd.read_csv("../data/raw/olist_orders_dataset.csv")
order_reviews = pd.read_csv("../data/raw/olist_order_reviews_dataset.csv")
order_payments = pd.read_csv("../data/raw/olist_order_payments_dataset.csv",
                             converters={"payment_value": to_decimal_or_none})
order_items = pd.read_csv("../data/raw/olist_order_items_dataset.csv", 
                          converters={"price": to_decimal_or_none, 
                                      "freight_value": to_decimal_or_none})
products = pd.read_csv("../data/raw/olist_products_dataset.csv")
sellers = pd.read_csv("../data/raw/olist_sellers_dataset.csv")
customers = pd.read_csv("../data/raw/olist_customers_dataset.csv")
geolocation = pd.read_csv("../data/raw/olist_geolocation_dataset.csv")
product_category_name_translation = pd.read_csv("../data/raw/product_category_name_translation.csv")

## 1. `orders` table

In [2]:
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [3]:
# Convert all timestamp columns into datetime -> intepret as UTC-3 -> convert to UTC
orders_ts_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date"
]

orders[orders_ts_cols] = orders[orders_ts_cols].apply(
    lambda col: pd.to_datetime(col, errors="coerce") # convert strings to datetime
    .dt.tz_localize("UTC-03:00") # treat datetime as UTC-3
    .dt.tz_convert("UTC") # convert from UTC-3 to UTC
)

In [4]:
# Convert date columns into date without timezone
orders["order_estimated_delivery_date"] = pd.to_datetime(orders["order_estimated_delivery_date"], errors="coerce").dt.date

In [5]:
# --------------------------------------
# Business Checks and Add in Flag Columns
# --------------------------------------
# Rule 1: Miniumum number of timestamps based on order_status
status_rules = {
    "created": ["order_purchase_timestamp"],
    "approved": ["order_purchase_timestamp", "order_approved_at"],
    "invoiced": ["order_purchase_timestamp", "order_approved_at"],
    "processing": ["order_purchase_timestamp", "order_approved_at"],
    "shipped": ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date"],
    "delivered": ["order_purchase_timestamp", "order_approved_at", "order_delivered_customer_date"],
}

def missing_required_ts(row):
    required = status_rules.get(row["order_status"], [])
    return any(pd.isna(row[col]) for col in required)

# Add in flag columns
orders["missing_required_timestamps"] = orders.apply(missing_required_ts, axis=1)

# Rule 2: Status aware timestamp ordering
def status_aware_ordering(row):
    if row["order_status"] in ["created"]:
        return True
    if row["order_status"] in ["approved", "invoiced", "processing"]:
        return row["order_purchase_timestamp"] <= row["order_approved_at"]
    if row["order_status"] == "shipped":
        return (
            row["order_purchase_timestamp"] <= row["order_approved_at"] <=
            row["order_delivered_carrier_date"]
        )
    if row["order_status"] == "delivered":
        return (
            row["order_purchase_timestamp"] <= row["order_approved_at"] <=
            row["order_delivered_carrier_date"] <=
            row["order_delivered_customer_date"]
        )
    return True

orders["status_aware_ordering"] = orders.apply(status_aware_ordering, axis=1)

# Rule 3: Marked as delivered but missing delivery date
orders["delivered_status_with_missing_timestamp"] = (
    (orders["order_status"] == "delivered") &
    (orders["order_delivered_customer_date"].isna())
)

# Rule 4: Delivered timestamp but status not delivered
orders["delivered_timestamp_with_incorrect_status"] = (
    orders["order_delivered_customer_date"].notna() &
    (orders["order_status"] != "delivered")
)

In [6]:
# Rename columns for clarity
orders.rename(columns={
    "order_purchase_timestamp": "purchase_timestamp",
    "order_approved_at": "approved_timestamp",
    "order_delivered_carrier_date": "carrier_received_timestamp",
    "order_delivered_customer_date": "order_delivered_timestamp",
    "order_estimated_delivery_date": "estimated_delivery_date"
}, inplace=True)

print(orders.shape)
print(orders.dtypes)
orders.head()

(99441, 12)
order_id                                                  object
customer_id                                               object
order_status                                              object
purchase_timestamp                           datetime64[ns, UTC]
approved_timestamp                           datetime64[ns, UTC]
carrier_received_timestamp                   datetime64[ns, UTC]
order_delivered_timestamp                    datetime64[ns, UTC]
estimated_delivery_date                                   object
missing_required_timestamps                                 bool
status_aware_ordering                                       bool
delivered_status_with_missing_timestamp                     bool
delivered_timestamp_with_incorrect_status                   bool
dtype: object


,order_id,customer_id,order_status,purchase_timestamp,approved_timestamp,carrier_received_timestamp,order_delivered_timestamp,estimated_delivery_date,missing_required_timestamps,status_aware_ordering,delivered_status_with_missing_timestamp,delivered_timestamp_with_incorrect_status
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 13:56:33+00:00,2017-10-02 14:07:15+00:00,2017-10-04 22:55:00+00:00,2017-10-11 00:25:13+00:00,2017-10-18,False,True,False,False
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 23:41:37+00:00,2018-07-26 06:24:27+00:00,2018-07-26 17:31:00+00:00,2018-08-07 18:27:45+00:00,2018-08-13,False,True,False,False
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 11:38:49+00:00,2018-08-08 11:55:23+00:00,2018-08-08 16:50:00+00:00,2018-08-17 21:06:29+00:00,2018-09-04,False,True,False,False
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 22:28:06+00:00,2017-11-18 22:45:59+00:00,2017-11-22 16:39:59+00:00,2017-12-02 03:28:42+00:00,2017-12-15,False,True,False,False
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-14 00:18:39+00:00,2018-02-14 01:20:29+00:00,2018-02-14 22:46:34+00:00,2018-02-16 21:17:02+00:00,2018-02-26,False,True,False,False


## 2. `order_reviews` table

In [7]:
order_reviews.head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


In [8]:
# Convert all timestamp columns into datetime -> intepret as UTC-3 -> convert to UTC
order_reviews_ts_cols = [
    "review_answer_timestamp"
]

order_reviews[order_reviews_ts_cols] = order_reviews[order_reviews_ts_cols].apply(
    lambda col: pd.to_datetime(col, errors="coerce") # convert strings to datetime
    .dt.tz_localize("UTC-03:00") # treat datetime as UTC-3
    .dt.tz_convert("UTC") # convert from UTC-3 to UTC
)

In [9]:
# Convert all date columns into date without timezone
order_reviews["review_creation_date"] = pd.to_datetime(order_reviews["review_creation_date"], errors="coerce").dt.date

In [10]:
# Rename columns for clarity
order_reviews.rename(columns={
    "review_comment_title": "review_title",
    "review_comment_message": "review_message",
    "review_creation_date": "review_date",
    "review_answer_timestamp": "review_timestamp"
}, inplace=True)

print(order_reviews.shape)
print(order_reviews.dtypes)
order_reviews.head()

(99224, 7)
review_id                        object
order_id                         object
review_score                      int64
review_title                     object
review_message                   object
review_date                      object
review_timestamp    datetime64[ns, UTC]
dtype: object


,review_id,order_id,review_score,review_title,review_message,review_date,review_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18,2018-01-19 00:46:59+00:00
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10,2018-03-11 06:05:13+00:00
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17,2018-02-18 17:36:24+00:00
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21,2017-04-22 01:02:06+00:00
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01,2018-03-02 13:26:53+00:00


## 3. `order_payments` table

In [11]:
order_payments.head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [12]:
print(order_payments.shape)
print(order_payments.dtypes)

(103886, 5)
order_id                object
payment_sequential       int64
payment_type            object
payment_installments     int64
payment_value           object
dtype: object


## `order_items` table

In [13]:
order_items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [14]:
# Convert all timestamp columns into datetime -> intepret as UTC-3 -> convert to UTC
order_items_ts_cols = [
    "shipping_limit_date"
]

order_items[order_items_ts_cols] = order_items[order_items_ts_cols].apply(
    lambda col: pd.to_datetime(col, errors="coerce") # convert strings to datetime
    .dt.tz_localize("UTC-03:00") # treat datetime as UTC-3
    .dt.tz_convert("UTC") # convert from UTC-3 to UTC
)

In [15]:
# Rename columns for clarity
order_items.rename(columns={
    "shipping_limit_date": "ship_out_deadline"
}, inplace=True)

print(order_items.shape)
print(order_items.dtypes)
order_items.head()

(112650, 7)
order_id                          object
order_item_id                      int64
product_id                        object
seller_id                         object
ship_out_deadline    datetime64[ns, UTC]
price                             object
freight_value                     object
dtype: object


,order_id,order_item_id,product_id,seller_id,ship_out_deadline,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 12:45:35+00:00,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 14:05:13+00:00,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 17:48:30+00:00,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 13:10:18+00:00,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 16:57:51+00:00,199.90,18.14


## 5. `products` table

In [16]:
products.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


In [17]:
# Convert the `product_category_name_translation` to a dictionary for easy lookup
category_translation_dict = dict(zip(
    product_category_name_translation["product_category_name"],
    product_category_name_translation["product_category_name_english"]
))

# Map the Portuguese category names to English using the translation dictionary
# If a category name does not exist in the dictionary, it will be mapped to NaN
products["product_category_name_english"] = products["product_category_name"].map(category_translation_dict)

# Drop the orginal Portuguese category name column and replace with English version
products = products[["product_id", "product_category_name_english", "product_name_lenght", "product_description_lenght", "product_photos_qty", "product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]]

In [18]:
# Rename columns for clarity
products.rename(columns={
    "product_category_name_english": "category_name",
    "product_name_lenght": "name_length",
    "product_description_lenght": "description_length",
    "product_photos_qty": "photos_quantity",
    "product_weight_g": "weight_g",
    "product_length_cm": "length_cm",
    "product_height_cm": "height_cm",
    "product_width_cm": "width_cm"
}, inplace=True)

print(products.shape)
print(products.dtypes)
products.head()

(32951, 9)
product_id             object
category_name          object
name_length           float64
description_length    float64
photos_quantity       float64
weight_g              float64
length_cm             float64
height_cm             float64
width_cm              float64
dtype: object


,product_id,category_name,name_length,description_length,photos_quantity,weight_g,length_cm,height_cm,width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumery,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,art,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,sports_leisure,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,baby,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,housewares,37.0,402.0,4.0,625.0,20.0,17.0,13.0


## 6. `sellers` table

In [19]:
sellers.head()

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


In [20]:
# Pad zip_code_prefix to 5 digits with leading zeros
sellers["seller_zip_code_prefix"] = sellers["seller_zip_code_prefix"].apply(lambda x: str(x).zfill(5))

In [21]:
# Function to replace non-standard special characters with standard characters in a string
def replace_char(city_name):
    city_name = re.sub(r'[ãââàáä]', 'a', city_name)
    city_name = re.sub(r'[íîì]', 'i', city_name)
    city_name = re.sub(r'[úûùü]', 'u', city_name)
    city_name = re.sub(r'[éêèë]', 'e', city_name)
    city_name = re.sub(r'[óõôòö]', 'o', city_name)
    city_name = re.sub(r'[ç]', 'c', city_name)
    return city_name

In [22]:
# Replace special characters in seller_city
sellers["seller_city"] = sellers["seller_city"].apply(replace_char)

In [23]:
# Rename columns for clarity
sellers.rename(columns={
    "seller_zip_code_prefix": "zip_code_prefix",
    "seller_city": "city",
    "seller_state": "state"
}, inplace=True)

print(sellers.shape)
print(sellers.dtypes)
sellers.head()

(3095, 4)
seller_id          object
zip_code_prefix    object
city               object
state              object
dtype: object


,seller_id,zip_code_prefix,city,state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,04195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


## 7. `customers` table

In [24]:
customers.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [25]:
# Pad zip_code_prefix to 5 digits with leading zeros
customers["customer_zip_code_prefix"] = customers["customer_zip_code_prefix"].apply(lambda x: str(x).zfill(5))

In [26]:
# Replace special characters in customer city
customers["customer_city"] = customers["customer_city"].apply(replace_char)

In [27]:
# Rename columns for clarity
customers.rename(columns={
    "customer_zip_code_prefix": "zip_code_prefix",
    "customer_city": "city",
    "customer_state": "state"
}, inplace=True)

print(customers.shape)
print(customers.dtypes)
customers.head()

(99441, 5)
customer_id           object
customer_unique_id    object
zip_code_prefix       object
city                  object
state                 object
dtype: object


,customer_id,customer_unique_id,zip_code_prefix,city,state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,09790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,01151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,08775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


## 8. `geolocation` table

In [28]:
geolocation.head()

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP


In [29]:
# Remove duplicates
geolocation = geolocation.drop_duplicates()

# Pad zip_code_prefix to 5 digits with leading zeros
geolocation["geolocation_zip_code_prefix"] = geolocation["geolocation_zip_code_prefix"].apply(lambda x: str(x).zfill(5))

# Replace special characters in geolocation_city
geolocation["geolocation_city"] = geolocation["geolocation_city"].apply(replace_char)

In [30]:
# Group by (zip_code_prefix, city, state) and take mean coordinates
geolocation_cleaned = (
    geolocation.groupby(["geolocation_zip_code_prefix", "geolocation_city", "geolocation_state"])
    .agg(
        {
            "geolocation_lat": "mean",
            "geolocation_lng": "mean"
        }
    )
    .reset_index()
)

# Rename for clarity
geolocation_cleaned.rename(columns={
    "geolocation_zip_code_prefix": "zip_code_prefix",
    "geolocation_lat": "lat",
    "geolocation_lng": "lng",
    "geolocation_city": "city",
    "geolocation_state": "state"
}, inplace=True)

print(geolocation_cleaned.shape)
print(geolocation_cleaned.dtypes)
geolocation_cleaned.head()

(19618, 5)
zip_code_prefix     object
city                object
state               object
lat                float64
lng                float64
dtype: object


,zip_code_prefix,city,state,lat,lng
0,01001,sao paulo,SP,-23.550227,-46.634039
1,01002,sao paulo,SP,-23.547657,-46.634991
2,01003,sao paulo,SP,-23.549000,-46.635582
3,01004,sao paulo,SP,-23.549829,-46.634792
4,01005,sao paulo,SP,-23.549547,-46.636406


In [31]:
# Save cleaned tables to staging area
os.makedirs("../data/staging", exist_ok=True)

# Save the tables in parquet format to preserve data types
orders.to_parquet("../data/staging/cleaned_orders.parquet", index=False, engine="pyarrow")
order_reviews.to_parquet("../data/staging/cleaned_order_reviews.parquet", index=False, engine="pyarrow")
order_payments.to_parquet("../data/staging/cleaned_order_payments.parquet", index=False, engine="pyarrow")
order_items.to_parquet("../data/staging/cleaned_order_items.parquet", index=False, engine="pyarrow")
products.to_parquet("../data/staging/cleaned_products.parquet", index=False, engine="pyarrow")
sellers.to_parquet("../data/staging/cleaned_sellers.parquet", index=False, engine="pyarrow")
customers.to_parquet("../data/staging/cleaned_customers.parquet", index=False, engine="pyarrow")
geolocation_cleaned.to_parquet("../data/staging/cleaned_geolocation.parquet", index=False, engine="pyarrow")